# Reconstruct a policy from its published rule

These are demonstration recommendations on a public dataset; no real reader's identity is present and no recommendation is personalized to a real person.

Decision date: 2026-09-27. The following outputs come from all three Open Bandit campaigns.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['dataset'])
print('Protocol:', e['protocol']['name'])


Population: {'name': 'goodbooks-10k complete catalog and ratings', 'backend': 'pinned public CSV -> parquet -> NumPy/SciPy exact scoring', 'source': 'https://github.com/zygmuntz/goodbooks-10k', 'source_commit': '6dd165b555a7b47b2dd36743a425776e641ff50c', 'license': 'CC BY-SA 4.0', 'source_rows': 5976479, 'catalog_size': 10000, 'interactions': 5976479, 'train_interactions': 4781183, 'test_interactions': 1195296, 'positive_training_interactions': 3286330, 'readers': 53424, 'evaluated_users': 5000}
Protocol: Global source-order holdout, full catalog


In [2]:
obd = e['ope']['open_bandit']
print(obd['status'])
print(pd.DataFrame([{'campaign': c['campaign'], 'actions': c['actions'], 'random_test': c['evaluation_rows'], 'bts_reference': c['bts_reference_rows'], 'bts_reward': c['on_policy_bts']['mean']} for c in obd['campaigns']]).to_string(index=False))
print(pd.DataFrame([{'campaign':r['campaign'], 'slot':r['position'], 'estimator':r['estimator'], 'estimate':r['estimate']['mean'], 'BTS_empirical':r['bts_reference']['mean'], 'delta':r['difference_from_bts']['mean']} for r in obd['rows'] if r['policy'] == 'Official prior BTS approximation']).to_string(index=False))


measured_six_file_bts_benchmark
campaign  actions  random_test  bts_reference  bts_reward
     all       80         5000           4964    0.003425
     men       34         5000           4804    0.005412
   women       46         5000           4955    0.005045
campaign  slot estimator  estimate  BTS_empirical     delta
     all     1       IPS  0.002516       0.000589  0.001928
     all     1     SNIPS  0.002827       0.000589  0.002238
     all     1        DM  0.002439       0.000589  0.001850
     all     1        DR  0.002644       0.000589  0.002055
     all     2       IPS  0.015844       0.005386  0.010458
     all     2     SNIPS  0.016030       0.005386  0.010644
     all     2        DM  0.003105       0.005386 -0.002281
     all     2        DR  0.015843       0.005386  0.010457
     all     3       IPS  0.000433       0.004389 -0.003956
     all     3     SNIPS  0.000462       0.004389 -0.003927
     all     3        DM  0.004163       0.004389 -0.000226
     all     3  

## Dead end retained: treating the frozen prior as the exact logged policy

The official benchmark supplies a prior-based target. The actual log has varying propensities within item and slot, so an exact match cannot be asserted. This diagnostic rejected that stronger interpretation. We use the official approximation and show the mismatch limitation.

In [3]:
for campaign in obd['campaigns']:
    diagnostic = campaign['logged_propensity_diagnostic']
    print(campaign['campaign'], diagnostic)
    assert diagnostic['max_distinct_values_within_item_position'] > 1
print('Monte Carlo diagnostics:', [{k:v for k,v in r.items() if k != 'probabilities'} for r in obd['monte_carlo']])
print('Source-reference arithmetic check:', e['ope']['crosscheck']['status'])


all {'max_distinct_values_within_item_position': 325, 'max_range_within_item_position': 0.95114, 'interpretation': 'Logged BTS propensities vary within item and position. The frozen upstream prior is a reproducible benchmark approximation, not an exact per-impression policy reconstruction.'}
men {'max_distinct_values_within_item_position': 541, 'max_range_within_item_position': 0.713565, 'interpretation': 'Logged BTS propensities vary within item and position. The frozen upstream prior is a reproducible benchmark approximation, not an exact per-impression policy reconstruction.'}
women {'max_distinct_values_within_item_position': 358, 'max_range_within_item_position': 0.96244, 'interpretation': 'Logged BTS propensities vary within item and position. The frozen upstream prior is a reproducible benchmark approximation, not an exact per-impression policy reconstruction.'}
Monte Carlo diagnostics: [{'campaign': 'all', 'simulations': 200000, 'seed': 12345, 'maximum_probability_se': 0.000962